# 03. Two-stage anchor-based: Faster R-CNN

1. **Sweep on fold 0:** the default model and four single-change experiments.
2. **Full cross-validation** of the best experiment (out-of-fold metrics); initial vs tuned is compared on fold 0.
3. **Analysis:** what each stage contributes, RPN proposals at test time, failure cases.

## Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/core").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q faster-coco-eval

In [ ]:
import pandas as pd

from core import N_FOLDS, PROTOCOL
from core.cv import compare, cv_table, environment, existing, load_oof, run_cv
from core.data import assign_folds, find_data_root, load_pool, pool_targets
from core.fusion import fold_ensemble
from core.errors import match_detections, outcome_tables, plot_failures, worst_frames
from core.submission import write_submission
from core.tv import load_model, make_tv_fns, plot_histories, proposal_sweep, two_stage_breakdown

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
environment()

## Data and folds

In [ ]:
DATA = find_data_root()
frames, gt = load_pool(DATA)
targets = pool_targets(gt)
frames["fold"] = frames["video"].map(assign_folds(frames))
frames.groupby("fold").agg(clips=("video", "nunique"), frames=("uid", "size"))

## Training setup
- **Model:** Faster R-CNN, ResNet-50-FPN v2, COCO-pretrained; new 8-class box predictor.
- **Input:** long side 1280 px, every 5th frame, ignored regions masked, horizontal flip.
- **Training:** SGD (momentum 0.9, weight decay 1e-4), lr 0.005, warm-up + cosine, mixed precision, 8 epochs.
- **Evaluation:** last epoch, all held-out frames; mAP, AP50, AP_small, inference speed (batch 1, T4).

Experiments, one change each against the default:
- **RoI sampling 50% positives** (default 25%): dense frames give enough foreground proposals.
- **Smaller RPN anchors, 16–256 px** (default 32–512): most objects are 16–64 px at 1280.
- **Box-regression loss ×2** (loss modification): more weight on localisation.
- **Step learning-rate schedule** (×0.1 at 2/3 and 8/9 of training) instead of cosine.

The number of RPN proposals is varied at test time on a trained model.

In [ ]:
BASE = dict(lr=0.005)
EXPERIMENTS = {
    "frcnn_default": BASE,
    "frcnn_roi_pos50": {**BASE, "roi_pos_fraction": 0.5},
    "frcnn_anchors": {**BASE, "anchor_sizes": (16, 32, 64, 128, 256)},
    "frcnn_boxloss2": {**BASE, "loss_weights": {"loss_box_reg": 2.0}},
    "frcnn_step_lr": {**BASE, "schedule": "step"},
}
SWEEP_FOLDS = [0]


def config(name):
    return {**PROTOCOL, **EXPERIMENTS[name], "name": name}


pd.Series(PROTOCOL)

## Sweep on fold 0

In [ ]:
for name in EXPERIMENTS:
    print(name, EXPERIMENTS[name])
    run_cv(frames, gt, config(name), *make_tv_fns(config(name), WORK, gt, targets), WORK, folds=SWEEP_FOLDS)

### Fold-0 comparison
Includes the YOLO baseline on the same held-out fold.

In [ ]:
sweep = compare([*EXPERIMENTS, "baseline"], WORK, fold=0)
display(sweep.round(4))
plot_histories(EXPERIMENTS, 0, WORK);

## Full cross-validation of the best experiment
Initial vs tuned is compared on fold 0 above; only the best experiment is run on all folds.
It is chosen on fold 0, so its fold-0 score is slightly optimistic.

In [ ]:
BEST = sweep.loc[sweep.index.isin(EXPERIMENTS), "mAP"].idxmax()
print("best on fold 0:", BEST)
result = run_cv(frames, gt, config(BEST), *make_tv_fns(config(BEST), WORK, gt, targets), WORK)

In [ ]:
display(compare([BEST, "baseline"], WORK).round(4))  # OOF over all frames
cv_table(result).round(4)

## What each stage contributes
- **RPN (stage 1)** turns anchors into class-agnostic proposals. Its job is recall: every object should be covered.
- **RoI head (stage 2)** classifies each proposal and refines its box. Its job is precision and localisation.

Measured on fold 0 by object size: share of objects covered (IoU ≥ 0.5) by the top-n proposals and by
the final detections, and the median best IoU before and after stage 2.

In [ ]:
cfg = config(BEST)
model = load_model(existing(f"{BEST}_fold0.pt", WORK), cfg)
held0 = frames[frames["fold"] == 0]
two_stage_breakdown(model, held0, targets, cfg)

### RPN proposals at test time

In [ ]:
proposal_sweep(model, held0.iloc[::5], gt, cfg)

## Failure analysis
Out-of-fold detections of the best model (score ≥ 0.3, IoU 0.5): missed, wrong class, poor box, duplicate, background.

In [ ]:
oof = load_oof(BEST, range(N_FOLDS), WORK)
objects, detections = match_detections(oof, frames, gt, score_thr=0.3)
by_size, by_class, det_share = outcome_tables(objects, detections)
display(by_size)
display(by_class)
det_share

In [ ]:
cases = worst_frames(objects, detections)
plot_failures(cases, objects, detections, frames, WORK / "failure_cases.png");

### Failure cases and plausible causes
*(To be written after the run, one paragraph per case.)*
1. **Most missed objects:** sizes, classes, density; were they covered by proposals?
2. **Most wrong-class objects:** which classes are confused and why.
3. **Most background false positives:** what the model fires on.

## Submission
The four fold models of the best experiment, fused by Weighted Boxes Fusion.

In [ ]:
test = pd.read_csv(DATA / "test_images.csv")
_, predict_fn = make_tv_fns(config(BEST), WORK, gt, targets)
test_dets = fold_ensemble(BEST, predict_fn, [str(DATA / f) for f in test["file_name"]], WORK)
write_submission(test_dets, test, WORK / f"submission_{BEST}.csv").head()